# 02 — Categorize Issues

## Goal
Run the categorization schema against the raw issue sample and produce a prioritized spreadsheet.

## 1. Setup

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))
from src.categorizer import categorize

DATA_DIR = Path('../data')
DELIVERABLES_DIR = Path('../deliverables')
DELIVERABLES_DIR.mkdir(exist_ok=True)

## 2. Load Raw Issues

In [2]:
issues = json.loads((DATA_DIR / 'raw_sample.json').read_text())
print(f'Loaded {len(issues)} issues')

Loaded 1000 issues


## 3. Categorize

In [3]:
rows = [categorize(i) for i in issues]
df = pd.DataFrame(rows)
print(f'Categorized {len(df)} issues')
df.head(3)

Categorized 1000 issues


,number,title,url,type,area,platform,priority,labels,comments,reactions,has_repro,created_at,updated_at,author
0,57169,[BUG] Plan files don't consistently open in re...,https://github.com/anthropics/claude-code/issu...,Bug,IDE Integration,VS Code,P3,"bug, area:ide, platform:vscode",1,0,False,2026-05-08,2026-05-08,sIlENtbuffER
1,57168,[BUG] Skill description budget uses base con...,https://github.com/anthropics/claude-code/issu...,Regression,Skills,macOS,P1,"bug, platform:macos, regression, area:skills",1,0,False,2026-05-08,2026-05-08,jadavenp
2,57167,[BUG] branch-listing widget — agent confirmed ...,https://github.com/anthropics/claude-code/issu...,Bug,TUI / Interface,Windows,P3,"bug, platform:windows, area:ui",1,0,False,2026-05-08,2026-05-08,gmanch94


## 4. Priority Distribution

In [4]:
print('Priority breakdown:')
print(df['priority'].value_counts().to_string())
print()
print('Type breakdown:')
print(df['type'].value_counts().to_string())
print()
print('Area breakdown:')
print(df['area'].value_counts().to_string())

Priority breakdown:
priority
P3    784
P2    155
P1     31
P0     30

Type breakdown:
type
Bug            603
Enhancement    255
Invalid         46
Duplicate       38
Other           30
Regression      28

Area breakdown:
area
TUI / Interface         138
Model / AI Behavior     123
Core / CLI              112
Cost / Token Usage       90
Other                    71
MCP                      67
Auth / Permissions       55
Cowork                   47
Desktop App              44
IDE Integration          32
Tools                    29
Agents                   26
Plugins                  25
Docs                     23
Hooks                    22
Installation / Setup     21
Skills                   19
Memory / Context         12
Sandbox                  10
API                      10
Security                  9
Networking                7
Chrome / Browser          6
Integrations              2


## 5. Top Issues by Engagement

In [5]:
df['engagement'] = df['reactions'] + df['comments']
top10 = (
    df.sort_values(['priority', 'engagement'], ascending=[True, False])
    .head(10)[['number', 'priority', 'type', 'area', 'engagement', 'title']]
)
print('Top 10 issues by priority + engagement:')
print(top10.to_string(index=False))

Top 10 issues by priority + engagement:
 number priority        type                area  engagement                                                                                               title
  56779       P0         Bug  Auth / Permissions           6                                                                                               [BUG]
  56255       P0         Bug               Tools           3        [Bug] Agent executes destructive database operations without confirmation, causing data loss
  56739       P0         Bug            Security           2 [BUG] without confirmation,ran find across entire Desktop and sent personal file to a 3rd-party API
  56639       P0         Bug         Desktop App           2                  Desktop: manual Archive deletes worktree with uncommitted changes, no confirmation
  56418       P0         Bug  Auth / Permissions           2                                          [BUG] Claude is calling bash commands that are prohib

## 6. Save Spreadsheet

In [6]:
out = DELIVERABLES_DIR / 'issue_tracker.xlsx'
df.sort_values(['priority', 'engagement'], ascending=[True, False]).to_excel(out, index=False)
print(f'Saved {len(df)} rows → {out}')

Saved 1000 rows → ../deliverables/issue_tracker.xlsx
